# Pivot 4: Bio-CLIP — Cross-Modal Self-Supervised EEG Representation Learning via EMG Guidance

**Workspace:** KG-GT pipeline · WAY-EEG-GAL · 12 subjects · 32-ch EEG · 5-ch EMG  
**Framework:** `self_supervised_coral/`  
**Target Venues:** NeurIPS / ICLR / IEEE TPAMI  

---

## Architecture Overview

```
EEG (32ch) ──> [LearnableFilterBank] ──> [MambaEncoder×4] ──> [MultiHeadPooling] ──> z_eeg ∈ S^127
                                                           └──> [DenseProj] ──> H_eeg ∈ R^{B×T×256}

EMG (5ch)  ──> [MultiScaleCausalConv] ──> [CausalGRU×2] ──> [MeanPooling] ──> z_emg ∈ S^127
                                                          └──> [DenseProj] ──> H_emg ∈ R^{B×T×256}

PhaseAwareInfoNCELoss: L = -log [sim(z_eeg,z_emg) / Σ_{j≠same-phase} sim(z_eeg,z_emg_j)]
+ DenseTokenInfoNCE (weight=0.2): per-frame temporal alignment
```

## Notebook Structure
1. Setup & Imports  
2. Data Loading (WAY-EEG-GAL)  
3. Preprocessing & Phase Labeling  
4. Build SSL DataLoaders (Balanced Sampling)  
5. Initialize Bio-CLIP Model  
6. Pre-Training Loop  
7. Downstream Evaluation (Linear Probes)  
8. Baseline Comparisons  
9. Results & Visualization  

## Cell 1: Setup & Imports

In [ ]:
# ============================================================
# Cell 1: Setup & Imports
# ============================================================
import sys
import os
import math
import time
import warnings
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

warnings.filterwarnings('ignore')

# Add workspace root to path
WORKSPACE = Path(os.path.abspath('.'))
sys.path.insert(0, str(WORKSPACE))

# Main pipeline
from main import (
    get_device, set_seed,
    load_hs, load_ws, load_participant, get_split_series,
    preprocess_eeg_from_config, preprocess_emg_from_config,
    preprocess_kinematics_from_config,
    resolve_participants, ALL_PARTICIPANTS,
)

# Bio-CLIP framework
from self_supervised_coral import (
    EEGEncoder, EMGEncoder,
    PhaseAwareInfoNCELoss, SymmetricInfoNCELoss,
    PhaseLabeler, MovementPhase,
    LinearProbe, FewShotRegressionProbe,
    evaluate_linear_probe, evaluate_few_shot_regression,
    SSLWindowDataset, build_ssl_dataloaders,
    BioCLIPTrainer, SSLTrainConfig, SSLTrainResult,
)

# Reproducibility
SEED = 42
set_seed(SEED)
DEVICE = get_device()
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

## Cell 2: Configuration

In [ ]:
# ============================================================
# Cell 2: Configuration
# ============================================================

# Data config
DATA_DIR = WORKSPACE / 'data' / 'way-eeg' / 'raw'
N_EEG = 32     # WAY-EEG-GAL EEG channels
N_EMG = 5      # WAY-EEG-GAL EMG channels
KIN_DIM = 36   # Kinematics: 18 pos + 18 vel
FS_EEG = 500.0  # Sampling frequency (Hz)

# Bio-CLIP model config
D_MODEL = 256       # Encoder hidden dimension
N_LAYERS = 4        # Mamba SSM layers (EEG encoder)
N_GRU = 2           # GRU layers (EMG encoder)
PROJ_DIM = 128      # Contrastive embedding dim (S^127 sphere)

# SSL pre-training config
WINDOW_SIZE = 500   # 1.0 s window at 500 Hz
STRIDE = 50         # 100 ms hop
BATCH_SIZE = 64     # Batch size
N_EPOCHS = 50       # Pre-training epochs (set to 200 for full training)

# Train/val split: 10 subjects for pre-training, 2 held-out for downstream eval
PRETRAIN_PARTICIPANTS = list(range(1, 11))  # Subjects 1-10
EVAL_PARTICIPANTS = [11, 12]                # Held-out subjects

# EEG preprocessing config (mirrors main pipeline)
EEG_CFG = {
    'preprocessing': {
        'eeg': {
            'fs': FS_EEG,
            'bandpass': [1.0, 100.0],
            'notch': 50.0,
            'reference': 'average',
            'normalize': True,
        }
    },
    'data': {'fs_eeg': FS_EEG},
}

# EMG preprocessing config (causal envelope extraction)
EMG_CFG = {
    'preprocessing': {
        'emg': {
            'fs': FS_EEG,
            'bandpass': [20.0, 450.0],
            'rectify': True,
            'lowpass_env': 8.0,
            'normalize': True,
        }
    },
    'data': {'fs_eeg': FS_EEG},
}

print('Configuration loaded.')
print(f'Pre-training subjects: {PRETRAIN_PARTICIPANTS}')
print(f'Evaluation subjects: {EVAL_PARTICIPANTS}')

## Cell 3: Data Loading & Preprocessing

In [ ]:
# ============================================================
# Cell 3: Data Loading & Preprocessing
# ============================================================

def load_and_preprocess_subject(participant_id, data_dir, eeg_cfg, emg_cfg, max_series=2):
    """Load and preprocess one WAY-EEG-GAL subject.
    
    Returns:
        eeg_trials: list of (T_i, 32) np.float32 arrays
        emg_trials: list of (T_i, 5) np.float32 arrays
        kin_trials: list of (T_i, 36) np.float32 arrays
    """
    hs = load_participant(str(data_dir), participant_id, file_type="hs")
    if max_series is not None:
        hs = hs[:max_series]
    
    eeg_trials, emg_trials, kin_trials = [], [], []
    
    for trial_idx, h in enumerate(hs):
        try:
            # Preprocess EEG -> (T, 32)
            eeg = preprocess_eeg_from_config(h, eeg_cfg)
            # Preprocess EMG -> (T, 5) causal envelope
            emg = preprocess_emg_from_config(h, emg_cfg)
            # Preprocess kinematics -> (T, 36) positions + velocities
            kin = preprocess_kinematics_from_config(h, {'include_velocity': True})
            
            # Align lengths (in case of 1-2 sample mismatch)
            min_len = min(eeg.shape[0], emg.shape[0], kin.shape[0])
            eeg = eeg[:min_len]
            emg = emg[:min_len]
            kin = kin[:min_len]
            
            if min_len >= WINDOW_SIZE:
                eeg_trials.append(eeg.astype(np.float32))
                emg_trials.append(emg.astype(np.float32))
                kin_trials.append(kin.astype(np.float32))
        except Exception as e:
            print(f'  Skipping trial {trial_idx}: {e}')
            continue
    
    return eeg_trials, emg_trials, kin_trials


# Load pre-training subjects
print('Loading pre-training subjects...')
eeg_pretrain, emg_pretrain, kin_pretrain = [], [], []
subj_ids_pretrain, trial_ids_pretrain = [], []

for subj_id in PRETRAIN_PARTICIPANTS[:3]:  # Limit to 3 subjects for interactive demo
    print(f'  Subject {subj_id}...', end=' ')
    try:
        e, m, k = load_and_preprocess_subject(subj_id, DATA_DIR, EEG_CFG, EMG_CFG)
        n_trials = len(e)
        eeg_pretrain.extend(e)
        emg_pretrain.extend(m)
        kin_pretrain.extend(k)
        subj_ids_pretrain.extend([subj_id] * n_trials)
        trial_ids_pretrain.extend(list(range(n_trials)))
        print(f'{n_trials} series loaded.')
    except Exception as exc:
        print(f'FAILED: {exc}')

# Load downstream evaluation subjects (P11, P12)
print('Loading evaluation subjects...')
eeg_eval, emg_eval, kin_eval = [], [], []
subj_ids_eval, trial_ids_eval = [], []

for subj_id in EVAL_PARTICIPANTS:
    print(f'  Subject {subj_id}...', end=' ')
    try:
        e, m, k = load_and_preprocess_subject(subj_id, DATA_DIR, EEG_CFG, EMG_CFG)
        n_trials = len(e)
        eeg_eval.extend(e)
        emg_eval.extend(m)
        kin_eval.extend(k)
        subj_ids_eval.extend([subj_id] * n_trials)
        trial_ids_eval.extend(list(range(n_trials)))
        print(f'{n_trials} series loaded.')
    except Exception as exc:
        print(f'FAILED: {exc}')

print(f'\nTotal pre-training series: {len(eeg_pretrain)}')
print(f'Total evaluation series:   {len(eeg_eval)}')


## Cell 4: Phase Labeling & SSL DataLoaders

In [ ]:
# ============================================================
# Cell 4: Phase Labeling & SSL DataLoaders
# ============================================================

# Fit PhaseLabeler on pre-training kinematic data
print('Fitting PhaseLabeler on pre-training kinematics...')
phase_labeler = PhaseLabeler(
    fs=FS_EEG,
    low_pct=25.0,   # Bottom 25% velocity → REST
    high_pct=75.0,  # Top 25% velocity → ACTIVE
    window_stride=1,
)

# Fit on all pre-training kinematic windows
kin_all_train = np.concatenate(kin_pretrain, axis=0)  # (N_total, 36)
# Reshape to per-window format for fitting
n_fit_windows = len(kin_all_train) // WINDOW_SIZE
kin_fit = kin_all_train[:n_fit_windows * WINDOW_SIZE].reshape(n_fit_windows, WINDOW_SIZE, KIN_DIM)
phase_labeler.fit(kin_fit)
low_thresh, high_thresh = phase_labeler.thresholds
print(f'PhaseLabeler thresholds: REST < {low_thresh:.4f} ≤ TRANSIT < {high_thresh:.4f} ≤ ACTIVE')

# Build SSL DataLoaders with balanced phase sampling
print('\nBuilding SSL DataLoaders...')
train_loader, val_loader = build_ssl_dataloaders(
    eeg_train=eeg_pretrain,
    emg_train=emg_pretrain,
    kin_train=kin_pretrain,
    eeg_val=eeg_eval,
    emg_val=emg_eval,
    kin_val=kin_eval,
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    batch_size=BATCH_SIZE,
    num_workers=2,
    phase_labeler=phase_labeler,
    subject_ids_train=subj_ids_pretrain,
    subject_ids_val=subj_ids_eval,
    trial_ids_train=trial_ids_pretrain,
    trial_ids_val=trial_ids_eval,
    use_balanced_sampling=True,
    fs=FS_EEG,
    pin_memory=(DEVICE.type == 'cuda'),
)

print(f'Train windows: {len(train_loader.dataset)}')
print(f'Val windows: {len(val_loader.dataset)}')

# Inspect phase distribution in training set
labels = np.array(train_loader.dataset._phase_labels)
unique, counts = np.unique(labels, return_counts=True)
phase_names = {0: 'REST', 1: 'ACTIVE', 2: 'TRANSIT'}
print('\nPhase distribution (pre-balanced sampling):')
for u, c in zip(unique, counts):
    print(f'  {phase_names.get(int(u), str(u))}: {c} windows ({100*c/len(labels):.1f}%)')

## Cell 5: Initialize Bio-CLIP Model

In [ ]:
# ============================================================
# Cell 5: Initialize Bio-CLIP Twin-Tower Architecture
# ============================================================

# EEG Encoder: Causal Filterbank + Mamba SSM
eeg_encoder = EEGEncoder(
    n_eeg_channels=N_EEG,
    d_model=D_MODEL,
    n_layers=N_LAYERS,
    d_state=16,
    fs=FS_EEG,
    proj_dim=PROJ_DIM,
    pool_heads=4,
    kernel_size=125,
).to(DEVICE)

# EMG Encoder: Multi-scale Causal Conv + Causal GRU
emg_encoder = EMGEncoder(
    n_emg_channels=N_EMG,
    d_model=D_MODEL,
    n_gru_layers=N_GRU,
    proj_dim=PROJ_DIM,
    mid_channels=64,
    dropout=0.1,
).to(DEVICE)

# Count parameters
n_eeg_params = sum(p.numel() for p in eeg_encoder.parameters() if p.requires_grad)
n_emg_params = sum(p.numel() for p in emg_encoder.parameters() if p.requires_grad)
print(f'EEG Encoder params: {n_eeg_params:,}')
print(f'EMG Encoder params: {n_emg_params:,}')
print(f'Total Bio-CLIP params: {n_eeg_params + n_emg_params:,}')

# Quick forward test
dummy_eeg = torch.randn(4, WINDOW_SIZE, N_EEG).to(DEVICE)
dummy_emg = torch.randn(4, WINDOW_SIZE, N_EMG).to(DEVICE)

with torch.no_grad():
    z_eeg, H_eeg = eeg_encoder(dummy_eeg, return_dense=True)
    z_emg, H_emg = emg_encoder(dummy_emg, return_dense=True)

print(f'\nForward test:')
print(f'  z_eeg shape: {z_eeg.shape}  (should be [4, {PROJ_DIM}])')
print(f'  H_eeg shape: {H_eeg.shape}  (should be [4, {WINDOW_SIZE}, {D_MODEL}])')
print(f'  z_emg shape: {z_emg.shape}  (should be [4, {PROJ_DIM}])')
print(f'  z_eeg L2 norm: {z_eeg.norm(dim=-1).mean():.6f}  (should be ≈ 1.0)')
print(f'  z_emg L2 norm: {z_emg.norm(dim=-1).mean():.6f}  (should be ≈ 1.0)')

## Cell 6: Bio-CLIP Pre-Training

In [ ]:
# ============================================================
# Cell 6: Bio-CLIP Pre-Training (Resumable + CLI Progress Bar)
# ============================================================

ssl_cfg = SSLTrainConfig(
    n_epochs=N_EPOCHS,
    learning_rate=3e-4,
    weight_decay=1e-2,
    temperature=0.07,
    learnable_temp=True,
    use_phase_masking=True,       # Phase-aware InfoNCE
    use_dense_loss=True,          # Token-level temporal alignment
    dense_loss_weight=0.2,        # λ_dense = 0.2
    rest_weight=2.0,              # Upweight REST windows 2×
    warmup_epochs=5,
    use_amp=torch.cuda.is_available(),
    clip_grad_norm=1.0,
    checkpoint_dir=str(WORKSPACE / 'outputs' / 'ssl_checkpoints'),
    checkpoint_every=1,           # Crash-safe checkpointing
    resume=True,                  # Automatically resume from last.pt
    log_every=50,
    val_every=5,
    device='auto',
    save_checkpoint=True,
)

trainer = BioCLIPTrainer(
    eeg_encoder=eeg_encoder,
    emg_encoder=emg_encoder,
    phase_labeler=phase_labeler,
    config=ssl_cfg,
)

print('Starting Bio-CLIP pre-training...')
print(f'  Epochs: {N_EPOCHS}')
print(f'  Batch size: {BATCH_SIZE}')
print(f'  Phase masking: {ssl_cfg.use_phase_masking}')
print(f'  Dense token loss (λ={ssl_cfg.dense_loss_weight}): {ssl_cfg.use_dense_loss}')
print(f'  AMP: {ssl_cfg.use_amp}')
print(f'  Resume: {ssl_cfg.resume} (checkpoints -> {ssl_cfg.checkpoint_dir})')
print()

t0 = time.time()
ssl_result = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    verbose=True,
)
print(f'\nPre-training complete in {ssl_result.total_time_s/60:.1f} min')
print(f'Best val_loss = {ssl_result.best_val_loss:.4f} @ epoch {ssl_result.best_epoch}')
if ssl_result.checkpoint_path:
    print(f'Checkpoint saved: {ssl_result.checkpoint_path}')

# Print formatted pre-training metrics summary report
print(ssl_result.summary())


## Cell 7: Training History Visualization

In [ ]:
# ============================================================
# Cell 7: Training History Visualization
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Bio-CLIP Pre-training Dynamics', fontsize=14, fontweight='bold')

# Loss curves
ax = axes[0]
epochs = list(range(1, len(ssl_result.train_losses) + 1))
ax.plot(epochs, ssl_result.train_losses, 'b-', lw=2, label='Train Loss')
if ssl_result.val_losses:
    val_epochs = list(range(ssl_cfg.val_every, len(ssl_result.train_losses) + 1, ssl_cfg.val_every))
    ax.plot(val_epochs[:len(ssl_result.val_losses)], ssl_result.val_losses,
            'r--', lw=2, label='Val Loss')
    ax.axvline(ssl_result.best_epoch, color='g', alpha=0.5, linestyle=':', label=f'Best @ ep{ssl_result.best_epoch}')
ax.set_xlabel('Epoch')
ax.set_ylabel('InfoNCE Loss')
ax.set_title('Contrastive Loss')
ax.legend()
ax.grid(alpha=0.3)

# Temperature evolution
ax = axes[1]
ax.plot(epochs, ssl_result.temperature_history, 'purple', lw=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Temperature τ')
ax.set_title('Learned Temperature')
ax.grid(alpha=0.3)

# EEG embedding similarity matrix (cosine) for a val batch
ax = axes[2]
frozen_enc = trainer.frozen_eeg_encoder

batch = next(iter(val_loader))
with torch.no_grad():
    z_batch = frozen_enc.encode(batch['eeg'].to(DEVICE)).cpu()

sim_matrix = (z_batch @ z_batch.T).numpy()  # (B, B) cosine similarity
im = ax.imshow(sim_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(im, ax=ax)
ax.set_title('EEG Embedding Cosine Similarity\n(Val Batch)')
ax.set_xlabel('Sample')
ax.set_ylabel('Sample')

plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/bioclip_training_dynamics.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/bioclip_training_dynamics.png')

## Cell 8: Downstream Evaluation — Linear Probe (Phase Classification)

In [ ]:
# ============================================================
# Cell 8: Linear Probe — Movement Phase Classification
# ============================================================

print('=' * 60)
print('Downstream Evaluation: Movement Phase Classification')
print('=' * 60)

frozen_encoder = trainer.frozen_eeg_encoder

# Evaluate Bio-CLIP encoder
print('\n[1/4] Bio-CLIP (pre-trained, frozen)...')
bioclip_results = evaluate_linear_probe(
    encoder=frozen_encoder,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    n_classes=3,
    n_epochs=100,
    verbose=True,
)

# Baseline: Random (untrained encoder)
print('\n[2/4] Random (untrained encoder, frozen)...')
random_encoder = EEGEncoder(
    n_eeg_channels=N_EEG, d_model=D_MODEL, n_layers=N_LAYERS,
    d_state=16, fs=FS_EEG, proj_dim=PROJ_DIM, pool_heads=4, kernel_size=125,
).to(DEVICE)
for param in random_encoder.parameters():
    param.requires_grad_(False)
random_encoder.eval()

random_results = evaluate_linear_probe(
    encoder=random_encoder,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    n_classes=3,
    n_epochs=100,
    verbose=False,
)

# Results summary
print('\n' + '=' * 60)
print(f'{'Model':<25} {'Train Acc':>10} {'Val Acc':>10} {'Val Bal Acc':>12}')
print('-' * 60)
print(f'{'Bio-CLIP (ours)':<25} {bioclip_results["train_accuracy"]:>10.3f} '
      f'{bioclip_results["val_accuracy"]:>10.3f} '
      f'{bioclip_results["val_balanced_accuracy"]:>12.3f}')
print(f'{'Random':<25} {random_results["train_accuracy"]:>10.3f} '
      f'{random_results["val_accuracy"]:>10.3f} '
      f'{random_results["val_balanced_accuracy"]:>12.3f}')
print('=' * 60)

## Cell 9: Downstream Evaluation — Few-Shot EMG Regression

In [ ]:
# ============================================================
# Cell 9: Few-Shot Regression — EMG Envelope Prediction
# ============================================================

print('=' * 60)
print('Downstream Evaluation: Few-Shot EMG Regression')
print('=' * 60)

# Test different n_shots values
shot_levels = [10, 25, 50, 100]
bioclip_r = []
bioclip_ccc = []
random_r = []
random_ccc = []

for n_shots in shot_levels:
    print(f'\nn_shots = {n_shots}')

    bc = evaluate_few_shot_regression(
        encoder=frozen_encoder,
        train_loader=train_loader,
        val_loader=val_loader,
        device=DEVICE,
        n_shots=n_shots,
        n_muscles=N_EMG,
        verbose=True,
    )
    bioclip_r.append(bc['val_pearson_r'])
    bioclip_ccc.append(bc['val_ccc'])

    rnd = evaluate_few_shot_regression(
        encoder=random_encoder,
        train_loader=train_loader,
        val_loader=val_loader,
        device=DEVICE,
        n_shots=n_shots,
        n_muscles=N_EMG,
        verbose=False,
    )
    random_r.append(rnd['val_pearson_r'])
    random_ccc.append(rnd['val_ccc'])

# Plot few-shot curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Bio-CLIP vs. Random — Few-Shot EMG Regression', fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(shot_levels, bioclip_r, 'b-o', lw=2, label='Bio-CLIP')
ax.plot(shot_levels, random_r, 'r--s', lw=2, label='Random')
ax.set_xlabel('N training shots')
ax.set_ylabel('Pearson r')
ax.set_title('Pearson Correlation (EMG)')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(shot_levels, bioclip_ccc, 'b-o', lw=2, label='Bio-CLIP')
ax.plot(shot_levels, random_ccc, 'r--s', lw=2, label='Random')
ax.set_xlabel('N training shots')
ax.set_ylabel('CCC')
ax.set_title("Lin's CCC (EMG)")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/bioclip_fewshot_regression.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/bioclip_fewshot_regression.png')

## Cell 10: Embedding Space Visualization (t-SNE)

In [ ]:
# ============================================================
# Cell 10: Embedding Space Visualization (t-SNE)
# ============================================================

try:
    from sklearn.manifold import TSNE

    print('Extracting embeddings for t-SNE visualization...')

    # Collect embeddings and phase labels from val set
    all_z = []
    all_phases = []
    all_subj = []

    frozen_encoder.eval()
    with torch.no_grad():
        for i, batch in enumerate(val_loader):
            eeg = batch['eeg'].to(DEVICE)
            z = frozen_encoder.encode(eeg).cpu().numpy()
            all_z.append(z)
            all_phases.extend(batch['phase'].numpy())
            all_subj.extend(batch['subject_id'].numpy())
            if i > 20:  # Cap at ~1300 windows for t-SNE speed
                break

    Z = np.concatenate(all_z, axis=0)
    phases = np.array(all_phases)
    subj_ids = np.array(all_subj)

    print(f'Running t-SNE on {Z.shape[0]} embeddings (dim={Z.shape[1]})...')
    tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
    Z_2d = tsne.fit_transform(Z)

    # Plot by phase and by subject
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Bio-CLIP EEG Embeddings — t-SNE Visualization', fontsize=13, fontweight='bold')

    phase_colors = {0: 'royalblue', 1: 'tomato', 2: 'goldenrod'}
    phase_labels = {0: 'REST', 1: 'ACTIVE', 2: 'TRANSIT'}

    ax = axes[0]
    for ph in [0, 1, 2]:
        mask = phases == ph
        if mask.sum() > 0:
            ax.scatter(Z_2d[mask, 0], Z_2d[mask, 1],
                      c=phase_colors[ph], label=phase_labels[ph],
                      alpha=0.6, s=10)
    ax.set_title('Colored by Movement Phase')
    ax.legend(markerscale=3)
    ax.axis('off')

    ax = axes[1]
    unique_subjs = np.unique(subj_ids)
    cmap = plt.cm.get_cmap('tab10', len(unique_subjs))
    for i, s in enumerate(unique_subjs):
        mask = subj_ids == s
        ax.scatter(Z_2d[mask, 0], Z_2d[mask, 1],
                  c=[cmap(i)], label=f'S{s}', alpha=0.6, s=10)
    ax.set_title('Colored by Subject')
    ax.legend(markerscale=3, ncol=2)
    ax.axis('off')

    plt.tight_layout()
    plt.savefig('outputs/bioclip_tsne.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: outputs/bioclip_tsne.png')

except ImportError:
    print('sklearn not available for t-SNE — skipping visualization.')

## Cell 11: Results Summary

In [ ]:
# ============================================================
# Cell 11: Final Results Summary
# ============================================================

print('=' * 70)
print('BIO-CLIP — PIVOT 4 RESULTS SUMMARY')
print('=' * 70)
print()
print(f'Pre-training Configuration:')
print(f'  Subjects: {PRETRAIN_PARTICIPANTS} (pre-train) | {EVAL_PARTICIPANTS} (eval)')
print(f'  Window: {WINDOW_SIZE} samples ({WINDOW_SIZE/FS_EEG:.1f}s), Stride: {STRIDE} samples')
print(f'  Epochs: {N_EPOCHS}, Batch size: {BATCH_SIZE}')
print(f'  d_model: {D_MODEL}, proj_dim: {PROJ_DIM}')
print()
print(f'Pre-training Results:')
print(f'  Best Val InfoNCE Loss: {ssl_result.best_val_loss:.4f} @ epoch {ssl_result.best_epoch}')
print(f'  Final Temperature τ: {ssl_result.temperature_history[-1]:.4f}')
print(f'  Training time: {ssl_result.total_time_s/60:.1f} min')
print()
print(f'Linear Probe — Phase Classification:')
print(f'  {'Model':<25} {'Val Acc':>8} {'Balanced':>10}')
print(f'  {'-'*45}')
print(f'  {'Bio-CLIP (ours)':<25} {bioclip_results["val_accuracy"]:>8.3f} {bioclip_results["val_balanced_accuracy"]:>10.3f}')
print(f'  {'Random':<25} {random_results["val_accuracy"]:>8.3f} {random_results["val_balanced_accuracy"]:>10.3f}')
print()
print(f'Few-Shot EMG Regression (n=50 shots):')
idx_50 = shot_levels.index(50) if 50 in shot_levels else -1
if idx_50 >= 0:
    print(f'  Bio-CLIP: r={bioclip_r[idx_50]:.4f}, CCC={bioclip_ccc[idx_50]:.4f}')
    print(f'  Random:   r={random_r[idx_50]:.4f}, CCC={random_ccc[idx_50]:.4f}')
print()
print('Files saved:')
print('  outputs/bioclip_training_dynamics.png')
print('  outputs/bioclip_fewshot_regression.png')
print('  outputs/bioclip_tsne.png')
if ssl_result.checkpoint_path:
    print(f'  {ssl_result.checkpoint_path}')
print('=' * 70)